# Catalog and evaluator profile

In [1]:
import json
import math
import re
import statistics
import sys
from collections import Counter, defaultdict
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "catalog.jsonl").is_file():
            return candidate
    raise FileNotFoundError("Could not locate data/catalog.jsonl")


project_root = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(project_root))

from evaluator.local_evaluator import (
    COLOR_RE,
    MATERIAL_RE,
    catalog_index,
    classify_constraint,
    coarse_category,
    materialize_hidden_fields,
    searchable_text,
)


def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


catalog = load_jsonl(project_root / "data" / "catalog.jsonl")
samples = load_jsonl(project_root / "data" / "public_set.jsonl")
_, categories_by_asin, products_by_asin = catalog_index(
    project_root / "data" / "catalog.jsonl"
)
targets = [products_by_asin[sample["ground_truth"]["parent_asin"]] for sample in samples]

print(f"Project root: {project_root}")
print(f"Catalog rows: {len(catalog):,}")
print(f"Public sessions: {len(samples):,}")

Project root: /Users/lamperriat/Documents/ntu/err402-search-agent
Catalog rows: 50,000
Public sessions: 200


In [2]:
def is_missing(value: object) -> bool:
    return value is None or value == "" or value == [] or value == {}


def percentile_summary(values: list[float]) -> dict:
    ordered = sorted(values)

    def percentile(fraction: float) -> float:
        return ordered[round(fraction * (len(ordered) - 1))]

    return {
        "count": len(ordered),
        "mean": round(statistics.fmean(ordered), 3),
        "min": ordered[0],
        "p10": percentile(0.10),
        "p25": percentile(0.25),
        "p50": percentile(0.50),
        "p75": percentile(0.75),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "p99": percentile(0.99),
        "max": ordered[-1],
    }


fields = sorted(set().union(*(product.keys() for product in catalog)))
print("Fields:", fields)
print("Unique parent_asin:", len({product["parent_asin"] for product in catalog}))
print("\nField missingness and non-missing types:")
for field in fields:
    values = [product.get(field) for product in catalog]
    missing_count = sum(is_missing(value) for value in values)
    types = Counter(type(value).__name__ for value in values if not is_missing(value))
    print(
        field,
        f"missing={missing_count:,} ({missing_count / len(catalog):.1%})",
        f"types={dict(types)}",
    )

print("\nText-field character lengths:")
for field in ("title", "features", "details", "description", "categories", "store"):
    lengths = []
    for product in catalog:
        value = product.get(field)
        if isinstance(value, list):
            text = " ".join(map(str, value))
        elif isinstance(value, dict):
            text = " ".join(f"{key} {item}" for key, item in value.items())
        else:
            text = "" if value is None else str(value)
        lengths.append(float(len(text)))
    print(field, percentile_summary(lengths))

Fields: ['average_rating', 'categories', 'description', 'details', 'features', 'parent_asin', 'price', 'rating_number', 'store', 'title']
Unique parent_asin: 50000

Field missingness and non-missing types:
average_rating missing=0 (0.0%) types={'float': 50000}
categories missing=0 (0.0%) types={'list': 50000}
description missing=23,887 (47.8%) types={'list': 26113}
details missing=1,670 (3.3%) types={'dict': 48330}
features missing=5,219 (10.4%) types={'list': 44781}
parent_asin missing=0 (0.0%) types={'str': 50000}
price missing=39,473 (78.9%) types={'float': 10410, 'str': 117}
rating_number missing=0 (0.0%) types={'int': 50000}
store missing=314 (0.6%) types={'str': 49686}
title missing=2 (0.0%) types={'str': 49998}

Text-field character lengths:
title {'count': 50000, 'mean': 75.899, 'min': 0.0, 'p10': 35.0, 'p25': 49.0, 'p50': 73.0, 'p75': 97.0, 'p90': 119.0, 'p95': 135.0, 'p99': 181.0, 'max': 290.0}
features {'count': 50000, 'mean': 393.711, 'min': 0.0, 'p10': 0.0, 'p25': 40.0, 'p

In [3]:
def normalize_price(value: object) -> tuple[float | None, str]:
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        number = float(value)
        if math.isfinite(number) and number >= 0:
            return number, "exact"
    if isinstance(value, str):
        match = re.fullmatch(r"from\s+([0-9]+(?:\.[0-9]+)?)", value.strip(), re.I)
        if match:
            return float(match.group(1)), "lower_bound"
    return None, "missing"


price_kinds = Counter()
normalized_prices = []
raw_string_prices = Counter()
for product in catalog:
    raw_price = product.get("price")
    if isinstance(raw_price, str):
        raw_string_prices[raw_price] += 1
    normalized_price, price_kind = normalize_price(raw_price)
    price_kinds[price_kind] += 1
    if normalized_price is not None:
        normalized_prices.append(normalized_price)

print("Price kinds:", price_kinds)
print("Raw string prices:", raw_string_prices)
print("Usable-price distribution:", percentile_summary(normalized_prices))

print("\nCatalog versus public targets:")
for field in ("average_rating", "rating_number"):
    print(field)
    print("  catalog:", percentile_summary([float(product[field]) for product in catalog]))
    print("  targets:", percentile_summary([float(product[field]) for product in targets]))
for label, products in (("catalog", catalog), ("targets", targets)):
    prices = [
        normalized
        for product in products
        if (normalized := normalize_price(product.get("price"))[0]) is not None
    ]
    print(
        f"{label} price coverage={len(prices) / len(products):.1%}",
        percentile_summary(prices),
    )

Price kinds: Counter({'missing': 39585, 'exact': 10410, 'lower_bound': 5})
Raw string prices: Counter({'—': 112, 'from 12.99': 1, 'from 12.46': 1, 'from 21.30': 1, 'from 8.98': 1, 'from 5.99': 1})
Usable-price distribution: {'count': 10415, 'mean': 45.144, 'min': 0.0, 'p10': 9.99, 'p25': 14.99, 'p50': 22.79, 'p75': 39.99, 'p90': 80.0, 'p95': 138.88, 'p99': 379.99, 'max': 4119.0}

Catalog versus public targets:
average_rating
  catalog: {'count': 50000, 'mean': 4.087, 'min': 1.0, 'p10': 3.2, 'p25': 3.8, 'p50': 4.2, 'p75': 4.6, 'p90': 5.0, 'p95': 5.0, 'p99': 5.0, 'max': 5.0}
  targets: {'count': 200, 'mean': 4.372, 'min': 3.5, 'p10': 4.0, 'p25': 4.3, 'p50': 4.4, 'p75': 4.6, 'p90': 4.6, 'p95': 4.7, 'p99': 4.8, 'max': 5.0}
rating_number
  catalog: {'count': 50000, 'mean': 241.409, 'min': 1.0, 'p10': 1.0, 'p25': 3.0, 'p50': 12.0, 'p75': 59.0, 'p90': 260.0, 'p95': 602.0, 'p99': 3327.0, 'max': 408371.0}
  targets: {'count': 200, 'mean': 16179.415, 'min': 1.0, 'p10': 186.0, 'p25': 986.0, 'p50'

People tend to choose the products of higher ratings and rating numbers. 

In [4]:
category_paths = Counter(tuple(map(str, product["categories"])) for product in catalog)
leaf_categories = Counter(str(product["categories"][-1]) for product in catalog)
coarse_categories = Counter(
    coarse_category([str(value) for value in product["categories"]])
    for product in catalog
)

print("Unique category paths:", len(category_paths))
print("Unique leaf categories:", len(leaf_categories))
print("Top category paths:")
for path, count in category_paths.most_common(25):
    print(f"{count:>5}  {' > '.join(path)}")
print("\nTop leaf categories:")
for leaf, count in leaf_categories.most_common(30):
    print(f"{count:>5}  {leaf}")

pool_sizes = {"full_path": [], "leaf": [], "evaluator_coarse": []}
for sample in samples:
    asin = sample["ground_truth"]["parent_asin"]
    path = tuple(categories_by_asin[asin])
    pool_sizes["full_path"].append(float(category_paths[path]))
    pool_sizes["leaf"].append(float(leaf_categories[path[-1]]))
    pool_sizes["evaluator_coarse"].append(
        float(coarse_categories[coarse_category(list(path))])
    )
print("\nCandidate-pool sizes for public targets:")
for name, sizes in pool_sizes.items():
    print(name, percentile_summary(sizes))

print("\nPrice coverage for the 20 largest leaf categories:")
for leaf, count in leaf_categories.most_common(20):
    matching = [product for product in catalog if str(product["categories"][-1]) == leaf]
    usable = sum(normalize_price(product.get("price"))[0] is not None for product in matching)
    print(f"{leaf}: n={count}, usable_price={usable / count:.1%}")

Unique category paths: 1628
Unique leaf categories: 800
Top category paths:
 1136  Clothing, Shoes & Jewelry > Westlake
 1040  Clothing, Shoes & Jewelry > Novelty & More > Clothing > Novelty > Men > Shirts > T-Shirts
  765  Clothing, Shoes & Jewelry > Women > Shoes
  680  Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses > T-Shirts
  672  Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses > Blouses & Button-Down Shirts
  659  Clothing, Shoes & Jewelry > Women > Clothing > Dresses > Casual
  630  Clothing, Shoes & Jewelry > Women > Shoes > Pumps
  577  Clothing, Shoes & Jewelry > Men > Watches > Wrist Watches
  574  Clothing, Shoes & Jewelry > Novelty & More > Clothing > Novelty > Women > Tops & Tees > T-Shirts
  567  Clothing, Shoes & Jewelry > Women > Shoes > Fashion Sneakers
  545  Clothing, Shoes & Jewelry > Women > Shoes > Sandals > Platforms & Wedges
  531  Clothing, Shoes & Jewelry > Women > Jewelry > Necklaces > Pendant Necklaces
  520  Clothin

In [5]:
detail_groups = {
    "department": {"Department", "Suggested Users"},
    "brand": {"Brand", "Brand Name", "Manufacturer"},
    "color": {"Color"},
    "material": {
        "Material", "Material Type", "Fabric Type",
        "Outer Material", "Inner Material", "Metal Type",
    },
    "size": {"Size"},
    "style": {"Style", "Fit Type", "Pattern", "Neck Style", "Sleeve Type", "Closure Type"},
    "use_case": {"Sport", "Sport Type", "Occasion", "Theme"},
    "special_feature": {"Special Feature", "Special Features"},
}

print("Structured detail-group coverage:")
for group, keys in detail_groups.items():
    covered = 0
    values = Counter()
    for product in catalog:
        details = product.get("details") or {}
        found = [
            str(details[key]).strip().lower()
            for key in keys
            if isinstance(details, dict) and not is_missing(details.get(key))
        ]
        if found:
            covered += 1
            values.update(found)
    print(
        group,
        f"coverage={covered / len(catalog):.1%}",
        f"unique={len(values):,}",
        f"top={values.most_common(8)}",
    )

searchable_corpora = [searchable_text(product) for product in catalog]
materials = Counter(
    match.group(1).lower()
    for text in searchable_corpora
    if (match := MATERIAL_RE.search(text))
)
colors = Counter(
    match.group(1).lower()
    for text in searchable_corpora
    if (match := COLOR_RE.search(text))
)
print("\nEvaluator-regex material coverage:", sum(materials.values()) / len(catalog), materials)
print("Evaluator-regex color coverage:", sum(colors.values()) / len(catalog), colors)

Structured detail-group coverage:
department coverage=88.2% unique=163 top=[('womens', 25607), ('mens', 10830), ('unisex-adult', 2072), ('girls', 1558), ('boys', 1350), ('unisex-child', 657), ('baby-girls', 608), ('baby-boys', 335)]
brand coverage=49.2% unique=10,868 top=[('adidas', 256), ('skechers', 254), ('clarks', 190), ('nike', 181), ('nine west', 175), ('puma', 162), ('amazon collection', 157), ('asics', 156)]
color coverage=4.9% unique=1,123 top=[('black', 407), ('silver', 145), ('white', 85), ('blue', 70), ('brown', 66), ('red', 59), ('gold', 46), ('pink', 35)]
material coverage=4.6% unique=584 top=[('leather', 362), ('polyester', 223), ('metal', 212), ('stainless steel', 145), ('cotton', 119), ('nylon', 94), ('faux leather', 70), ('wood', 46)]
size coverage=1.8% unique=327 top=[('one size', 141), ('large', 86), ('medium', 82), ('small', 52), ('x-large', 41), ('xx-large', 19), ('8 inch', 12), ('one_size', 11)]
style coverage=4.2% unique=984 top=[('solid', 262), ('modern', 149),

In [6]:
hard_types = Counter()
soft_types = Counter()
first_hard_types = Counter()
cards_containing = Counter()
distinct_types_per_card = Counter()

for sample in samples:
    card, _ = materialize_hidden_fields(sample, products_by_asin)
    hard = [classify_constraint(str(value)) for value in card["hard_constraints"]]
    soft = [classify_constraint(str(value)) for value in card["soft_preferences"]]
    hard_types.update(hard)
    soft_types.update(soft)
    if hard:
        first_hard_types[hard[0]] += 1
    kinds = set(hard + soft)
    cards_containing.update(kinds)
    distinct_types_per_card[len(kinds)] += 1

print("Hard-constraint types:", hard_types)
print("Soft-preference types:", soft_types)
print("First hard-constraint type:", first_hard_types)
print("Cards containing each type:", cards_containing)
print("Distinct attribute types per card:", distinct_types_per_card)
print(
    "Profile tags:",
    Counter(tag for sample in samples for tag in sample["user_profile"]["preference_tags"]),
)

Hard-constraint types: Counter({'material': 258, 'feature': 90, 'color': 44, 'style': 5, 'size': 2, 'use_case': 1})
Soft-preference types: Counter({'feature': 314, 'material': 44, 'color': 16, 'style': 14, 'size': 9, 'use_case': 3})
First hard-constraint type: Counter({'material': 153, 'feature': 42, 'style': 4, 'use_case': 1})
Cards containing each type: Counter({'feature': 192, 'material': 153, 'color': 51, 'style': 18, 'size': 9, 'use_case': 4})
Distinct attribute types per card: Counter({2: 137, 3: 45, 1: 18})
Profile tags: Counter({'fit': 163, 'material': 154, 'comfort': 144, 'style': 101, 'durability': 47, 'performance': 26, 'warmth': 18, 'weather': 12, 'general shopping': 1})


## Interpretation checklist

When comparing outputs, check these design-relevant facts:

- Category paths are complete and much denser than most product attributes.
- Treat missing price as unknown, not zero. A `from X` value is a lower bound, not an exact price.
- `Department` is the only broadly covered structured shopping attribute; most other useful `details` fields are sparse.
- Material and color become substantially more available when extracted from text.
- Public targets are much more popular and much more likely to have prices than the catalog overall, so rating-count and price features can encode benchmark-selection bias.
- The current evaluator's generated cards overwhelmingly disclose material and generic feature text; they never expose budget or brand in the public set.